<a href="https://colab.research.google.com/github/SafaaMahbub/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-23%20%E2%80%94%20Cleaning%20Clinic%20%E2%80%94%20Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Clinic

**Studio — 2026-09-23 · Fall 2026**  
**Class time:** 45 minutes

---

## Write the pipeline, then defend it

Monday I made the cleaning decisions and told you what they were. Today you make them, and the output is two things: a clean frame, and a **decision log** that says what you did to whose rows and why.

The log is not paperwork. On the midterm your team will disagree about whether a refund counts, and the log is what turns that into a two-minute conversation instead of an afternoon of re-deriving numbers.

Every step below follows the same three-part shape: **do it, count what you changed, log the decision.**

In [1]:
import pandas as pd, numpy as np
from io import StringIO
raw = '''order_id,item,category,qty,price,ts
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
2,cheese burger,food,1,7.5,09/05/2026 12:40
3,Foam Finger,Merch,NULL,12,2026-09-05 13:00:00
4,UVA T-Shirt ,Apparel,2,$24.00,2026-09-05 13:05
5,Rain Poncho,RainGear,-3,6,2026-09-05T13:20:00
6,rain poncho,rain-gear,4,$6.00,
7,,Merch,1,12,2026-09-05T14:00:00'''
df = pd.read_csv(StringIO(raw))
df

,order_id,item,category,qty,price,ts
0,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
1,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
2,2,cheese burger,food,1.0,7.5,09/05/2026 12:40
3,3,Foam Finger,Merch,NaN,12,2026-09-05 13:00:00
4,4,UVA T-Shirt,Apparel,2.0,$24.00,2026-09-05 13:05
5,5,Rain Poncho,RainGear,-3.0,6,2026-09-05T13:20:00
6,6,rain poncho,rain-gear,4.0,$6.00,NaN
7,7,NaN,Merch,1.0,12,2026-09-05T14:00:00


### Set up the log

Run this first. Each step calls `log()` with what happened and how many rows it touched.

In [2]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)

raw_rows = len(df)
print('starting with', raw_rows, 'rows')

starting with 8 rows


### Step 0 — take inventory

**TODO:** print the shape, the dtypes, the null count per column, and the number of exact duplicate rows. Do not skip this — the rest of the studio depends on knowing what you have.

In [3]:
# TODO
print('Shape: ',df.shape)
print('Types: ')
print(df.dtypes)
print('')
print(df.isnull().sum())

print("\nduplicate rows:")
print(df.duplicated().sum())

Shape:  (8, 6)
Types: 
order_id      int64
item         object
category     object
qty         float64
price        object
ts           object
dtype: object

order_id    0
item        1
category    0
qty         1
price       0
ts          1
dtype: int64

duplicate rows:
1


**What is wrong with this data?** List at least five specific problems:

1. price is type of object although it should be an int/float type
2. qty column is of type float
3. Some of the rows for price has the $ sign and some dont have and therefore the column is showing inconsistent types
4. There are several rows that have null vlaues including the columns item, qty, and ts. ts should be automatically populated the moment an order is placed
5. There is redundant rows in the data which shows not uniqueness and redundant information

### Step 1 — duplicates

**TODO:** drop exact duplicate rows into a new frame called `clean`, then log how many you removed. Use `.copy()` so later assignments do not warn.

In [4]:
removed = df.duplicated().sum()   # TODO: how many duplicates were there?
clean = df.drop_duplicates().copy()    # TODO: df with duplicates dropped, copied

log('duplicates', 'dropped exact duplicate rows', removed)

[duplicates] dropped exact duplicate rows (1 row(s))


In [5]:
clean

,order_id,item,category,qty,price,ts
0,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
2,2,cheese burger,food,1.0,7.5,09/05/2026 12:40
3,3,Foam Finger,Merch,NaN,12,2026-09-05 13:00:00
4,4,UVA T-Shirt,Apparel,2.0,$24.00,2026-09-05 13:05
5,5,Rain Poncho,RainGear,-3.0,6,2026-09-05T13:20:00
6,6,rain poncho,rain-gear,4.0,$6.00,NaN
7,7,NaN,Merch,1.0,12,2026-09-05T14:00:00


### Step 2 — price into a real number

**TODO:** strip the dollar signs and any stray whitespace, then convert to float. Assert the dtype afterward so you find out now if a stray character survived.

In [6]:
clean['price'] = (clean['price'].astype(str)
                  .str.replace('$', '', regex=False)
                  .str.replace(',', '', regex=False)
                  .str.strip()
                  .astype(float))
clean['price'] = pd.to_numeric(clean['price'], errors='coerce')
assert clean['price'].dtype == float
# TODO: log(...) -- note that price arrived as text
log('price', 'extracting the $ and translate the type to an float', len(clean['price']))

[price] extracting the $ and translate the type to an float (7 row(s))


some of the rows in the price column had a $ character that has to be stripped away so that the string can be translated into a float value.

In [7]:
before_revenue = (clean['price']*clean['qty']).sum()
print('initial revenue: ', before_revenue)

initial revenue:  88.5


### Step 3 — quantity, and two decisions

**TODO:** coerce `qty` to numeric. Then decide, separately:

- what to do with the row that has no quantity
- what to do with the refund (negative quantity)

Log each decision with its row count. There is no single right answer — there is only an answer you can defend.

In [8]:
# TODO: clean['qty'] = pd.to_numeric(...)
clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce')
missing = clean['qty'].isna().sum()    # TODO: count of NaN quantities
negative = (clean['qty'] < 0).sum()   # TODO: count of negative quantities

# TODO: apply your decision, then log both separately
clean = clean[clean['qty'].notna()].copy()
clean['qty'] = clean['qty'].abs()
clean['qty'] = clean['qty'].astype(int)


log('qty','dropped null values from the qty column', missing)
log('qty','translating negative quantity to the absolute quantity', negative)

[qty] dropped null values from the qty column (1 row(s))
[qty] translating negative quantity to the absolute quantity (1 row(s))


I decided that to drop the na qty values because it will not change the revenue at all but I made sure to change the negative values for qty to become positive to get the actual overall gross period before returns.

In [9]:
after_revenue = (clean['price']*clean['qty']).sum()
print('initial revenue: ', after_revenue)
clean

initial revenue:  124.5


,order_id,item,category,qty,price,ts
0,1,Cheeseburger,Food,2,7.5,2026-09-05T12:03:00
2,2,cheese burger,food,1,7.5,09/05/2026 12:40
4,4,UVA T-Shirt,Apparel,2,24.0,2026-09-05 13:05
5,5,Rain Poncho,RainGear,3,6.0,2026-09-05T13:20:00
6,6,rain poncho,rain-gear,4,6.0,NaN
7,7,NaN,Merch,1,12.0,2026-09-05T14:00:00


### Step 4 — categories that mean one thing

**TODO:** normalize case and punctuation, then map the remaining variants with an explicit dict. Print the unique values before and after so the collapse is visible. Log how many distinct categories you started and ended with.

In [10]:
print('before:', sorted(clean['category'].unique()))

# TODO: lowercase, strip, remove punctuation
clean['category'] = (clean['category'].str.strip().str.lower() .str.replace('-', '', regex=False))
# TODO: CATEGORY_MAP = {...} for the judgment calls
CATEGORY_MAP = {
    'apparel': 'merch',
    'raingear': 'raingear',
}
print('after: ', sorted(clean['category'].unique()))
unique_values = clean['category'].nunique()
log('category','mapped vairants with an explicit dict', unique_values)

before: ['Apparel', 'Food', 'Merch', 'RainGear', 'food', 'rain-gear']
after:  ['apparel', 'food', 'merch', 'raingear']
[category] mapped vairants with an explicit dict (4 row(s))


Food and food are the same thing and so to just have one version of variants of the same word, I decided to map the current vlaues to an explicit dictionary map.

### Step 5 — item names

**TODO:** same treatment for `item`. One product is spelled two ways, and one row has no item at all — decide what to do with it.

In [11]:
# TODO
clean['item'] = clean['item'].str.strip().str.title()
ITEMMAP={
    'Cheese Burger': 'Cheeseburger',
}
clean['item'] = clean['item'].replace(ITEMMAP)
empty_value = clean['item'].isna().sum()

clean = clean.dropna(subset=['item']).copy()

log('item', 'I cleaned the item name to be consistent', empty_value)

[item] I cleaned the item name to be consistent (1 row(s))


I chose to just drop the empty value row as it couldnt be categorized properly.

### Step 6 — timestamps

**TODO:** parse `ts` into real datetimes, coercing failures to `NaT`. Report how many failed. Then add an `hour` column, which is only possible once the column is a real datetime.

In [12]:
# TODO
clean['ts'] = pd.to_datetime(clean['ts'],errors='coerce', format ='mixed')
missing_date = clean['ts'].isna().sum()
print('count failed timestamps :',missing_date)


clean['hour'] = clean['ts'].dt.hour
log('ts','parse into realtimes inlcuding translates failures to NaT',missing_date)



count failed timestamps : 1
[ts] parse into realtimes inlcuding translates failures to NaT (1 row(s))


Since there was a null value in the ts column, I decided to change it to NaT to make sure  address it. The hours well only applied properly after that

### Step 7 — prove it

**TODO:** write at least five assertions that would catch a regression in this pipeline. Then compute `revenue` and print the totals.

In [13]:
# TODO: assertions
assert clean['price'].notna().all()
assert (clean['qty'] >0).all()
assert clean['price'].dtype == float
assert clean.duplicated().sum() == 0
assert clean['category'].str.islower().all()
clean['revenue'] = clean['qty']*clean['price']
# TODO: print rows, units, revenue, distinct categories

print(f'rows:        {len(df)} raw -> {len(clean)} clean')
print(f'units:       {pd.to_numeric(df['qty'], errors='coerce').fillna(0).sum():.0f} raw -> {clean["qty"].sum()} clean')
print(f'categories:  {df['category'].nunique()} raw -> {clean["category"].nunique()} clean')
print(f'revenue:     ${clean["revenue"].sum():.2f} (not computable before cleaning)')
clean

rows:        8 raw -> 5 clean
units:       9 raw -> 12 clean
categories:  6 raw -> 3 clean
revenue:     $112.50 (not computable before cleaning)


,order_id,item,category,qty,price,ts,hour,revenue
0,1,Cheeseburger,food,2,7.5,2026-09-05 12:03:00,12.0,15.0
2,2,Cheeseburger,food,1,7.5,2026-09-05 12:40:00,12.0,7.5
4,4,Uva T-Shirt,apparel,2,24.0,2026-09-05 13:05:00,13.0,48.0
5,5,Rain Poncho,raingear,3,6.0,2026-09-05 13:20:00,13.0,18.0
6,6,Rain Poncho,raingear,4,6.0,NaT,NaN,24.0


### Step 8 — the decision log

**TODO:** print your log. Then answer, in the markdown cell below: which single decision moved your revenue total the most, and what is the number both ways?

In [14]:
show_log()

,step,decision,rows
0,duplicates,dropped exact duplicate rows,1
1,price,extracting the $ and translate the type to an ...,7
2,qty,dropped null values from the qty column,1
3,qty,translating negative quantity to the absolute ...,1
4,category,mapped vairants with an explicit dict,4
5,item,I cleaned the item name to be consistent,1
6,ts,parse into realtimes inlcuding translates fail...,1


**The decision that mattered most:** the decision that alternated the most rrevenue would be to changing negative qty to absolute value of the qty. I changed it to be the absolute value (from -3 to 3) because I wanted to care about the gross revenue and i still counted the refund as something being sold

**Revenue with it:** 124.5  **Revenue without it:**88.5

---

## Checkpoint (participation)

Report your row count and revenue after cleaning, and the one decision that moved the total most.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [15]:
# Checkpoint
rows_after = len(clean)            # TODO
revenue_after = after_revenue        # TODO
biggest_decision = 'step 3'    # TODO: which choice moved the number most
revenue_other_way = before_revenue     # TODO: the total if you had chosen differently

print('rows after cleaning:', rows_after)
print('revenue:', revenue_after)
print('decision that mattered:', biggest_decision)
print('revenue the other way:', revenue_other_way)

rows after cleaning: 5
revenue: 124.5
decision that mattered: step 3
revenue the other way: 88.5
